# Delta Lake

In [0]:
# Default Catalog for Databricks
spark.conf.get("spark.sql.catalogImplementation")

In [0]:
%sql
show databases

In [0]:
# Read Sales parquet data
df_sales = spark.read.parquet("/Volumes/workspace/pyspark/input/sales_data.parquet")

df_sales.display()

In [0]:
# Write Sales data as Delta Table
df_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.pyspark.sales_delta")

In [0]:
%sql
show tables in pyspark

In [0]:
%sql
describe extended workspace.pyspark.sales_delta

In [0]:
%sql
update workspace.pyspark.sales_delta set amount = 0 where trx_id = '1734117021'

In [0]:
%sql
select * from workspace.pyspark.sales_delta where trx_id = '1734117021'

In [0]:
# Read a particular version - pyspark api
df_sales_delta = spark.read.table("sales_delta@v1")

display(df_sales_delta.where("trx_id = '1734117021'"))

In [0]:
%sql

select * from sales_delta@v0 where trx_id = '1734117021'

In [0]:
df_new = spark.sql("select *, current_timestamp() as time_now from workspace.pyspark.sales_delta where trx_id = '1734117021'")

display(df_new)

In [0]:
# Append data to existing delta table
df_new.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("workspace.pyspark.sales_delta")



In [0]:
# Append data to existing delta table
df_new.write.format("delta").mode("append").option("mergeSchema", True).option("path", "/tables/pyspark/sales_delta_1/").saveAsTable("sales_delta")
     


In [0]:
# Reading Delta Table using Delta libraries
from delta import DeltaTable

dt = DeltaTable.forName(spark, "sales_delta")

display(dt.history())

In [0]:
# Converting a Parquet to Delta - Check and Convert

DeltaTable.isDeltaTable(spark, "/data/output/sales_delta_1")

DeltaTable.convertToDelta(spark, "parquet.`/data/output/sales_parquet_1`")

In [0]:
# Validate
display(dbutils.fs.ls("/data/output/sales_parquet_1"))

In [0]:
%sql

describe extended sales_parquet

In [0]:
%sql

CONVERT TO DELTA sales_parquet

In [0]:
%sql
RESTORE TABLE sales_delta TO VERSION AS OF 1

In [0]:
%sql

select * from sales_delta where trx_id = '1734117021'

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled","false")

dt = DeltaTable.forName(spark, "sales_delta")

dt.vacuum(0)

In [0]:
display(dbutils.fs.ls("/data/output/sales_delta_1"))

In [0]:
%sql

select * from sales_delta@v0 where trx_id = '1734117021'